In [ ]:
import os
import sys
import torch
import numpy as np
import matplotlib.pyplot as plt

sys.path.append(os.path.abspath('..'))

from src.utils import load_full_config
from src.model import SimpleTransformer


In [ ]:
def fit_parabola(s_range, phi_values):
    X = np.stack([s_range ** 2, s_range], axis=1)
    coeffs, _, _, _ = np.linalg.lstsq(X, phi_values, rcond=None)
    A, B = coeffs
    return A, B


In [ ]:
cfg = load_full_config()

device = torch.device(
    'cuda' if torch.cuda.is_available() and cfg['system']['device'] == 'cuda' else 'cpu'
)

# Load data
data_path = os.path.join('..', cfg['paths']['mgf_data_path'])
data = torch.load(data_path, weights_only=False)

trajectories = data['trajectories']
targets = data['targets']
theta_values = data['theta_values']
s_range = data['s_range']

# Load model
model = SimpleTransformer(**cfg['architecture'])
model_path = os.path.join(
    '..',
    cfg['paths']['save_dir'],
    cfg['paths'].get('mgf_model_name', 'model_mgf.pth')
)

if os.path.exists(model_path):
    model.load_state_dict(torch.load(model_path, map_location=device))
    model.to(device)
    model.eval()
    print(f"Loaded model from {model_path}")
else:
    raise FileNotFoundError(
        f"Model not found at {model_path}. Train first with python ../scripts/train.py"
    )


In [ ]:
num_test = 300
indices = np.random.choice(len(trajectories), num_test, replace=False)

batch_trajs = trajectories[indices].to(device)
batch_thetas = theta_values[indices].numpy()
X_Ls = batch_trajs[:, -1, 0].cpu().numpy()

with torch.no_grad():
    preds, _ = model(batch_trajs)
    phi_preds = preds[:, -1, :].cpu().numpy()

pred_means = np.zeros(num_test)
pred_vars = np.zeros(num_test)
for i in range(num_test):
    A, B = fit_parabola(s_range, phi_preds[i])
    pred_means[i] = B
    pred_vars[i] = 2 * A

D = cfg['physics']['D']
dt = cfg['physics']['dt']
mu = cfg['physics']['mu']

exp_theta_dt = np.exp(-batch_thetas * dt)
theory_means = exp_theta_dt * X_Ls + mu * (1 - exp_theta_dt)
theory_vars = (D / batch_thetas) * (1 - np.exp(-2 * batch_thetas * dt))

fig, axes = plt.subplots(1, 2, figsize=(12, 5))

ax = axes[0]
ax.scatter(theory_means, pred_means, alpha=0.6, s=25)
min_val = min(theory_means.min(), pred_means.min())
max_val = max(theory_means.max(), pred_means.max())
ax.plot([min_val, max_val], [min_val, max_val], 'k--', alpha=0.6)
ax.set_xlabel('Theoretical Mean')
ax.set_ylabel('Predicted Mean')
ax.set_title('Mean Regression')
ax.grid(True, alpha=0.3)
ax.set_aspect('equal')

ax = axes[1]
ax.scatter(theory_vars, pred_vars, alpha=0.6, s=25)
min_val = min(theory_vars.min(), pred_vars.min())
max_val = max(theory_vars.max(), pred_vars.max())
ax.plot([min_val, max_val], [min_val, max_val], 'k--', alpha=0.6)
ax.set_xlabel('Theoretical Variance')
ax.set_ylabel('Predicted Variance')
ax.set_title('Variance Regression')
ax.grid(True, alpha=0.3)
ax.set_aspect('equal')

plt.tight_layout()
plt.show()
